In [1]:
from oqd_core.interface.analog import MathAdd, MathNum

a = MathNum(value=1) + MathNum(value=2)

print(a)

s = a.model_dump_json()

b = MathAdd.model_validate_json(s)

print(b)

class_='MathAdd' expr1=MathNum(class_='MathNum', value=1) expr2=MathNum(class_='MathNum', value=2)
class_='MathAdd' expr1=MathNum(class_='MathNum', value=1) expr2=MathNum(class_='MathNum', value=2)


In [2]:
from oqd_compiler_infrastructure import CFGBlockAccumulator, Post, PrettyPrint

from oqd_core.analysis.analog.cfg import AnalogCFGBuilder
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker
from oqd_core.frontend.analog import parse_analog

printer = Post(PrettyPrint())

with open("test.analog", mode="r", encoding="utf8") as f:
    source = f.read()

circuit = parse_analog(source)
cfg = AnalogCFGBuilder()(circuit)
cfg = CFGBlockAccumulator()(cfg)
checker = AnalogTypeChecker(cfg)


In [3]:
# from oqd_core.frontend.analog import parse_analog

# program = "r = qreg(2) \n H_single = 0.5 %* %X %+ 0.5 %* %Z \n evolve(H_single, 1.0, r[0])"
# circuit = parse_analog(program)

In [4]:
import json

from oqd_core.analysis.analog.symbol_table import AnalogSymbolTableBuilder

symbol_analysis = AnalogSymbolTableBuilder(cfg, checker.dataflow_result)
symbol_table = symbol_analysis.symbol_table


symbol_table = str(symbol_table)
print(json.dumps(symbol_table, indent=2))

"in_env={0: {}, 1: {}, 26: {'r': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TQReg'>, target_dim=2, list_elem=None), 's': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TMReg'>, target_dim=3, list_elem=None), 'q0': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TTargetRef'>, target_dim=1, list_elem=None), 'q1': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TTargetRef'>, target_dim=1, list_elem=None), 'targets': SymbolBinding(lattice_type=TList(elem=<class 'oqd_core.analysis.analog.types.TQRef'>), target_dim=2, list_elem=SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TTargetRef'>, target_dim=1, list_elem=None))}, 27: {'r': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TQReg'>, target_dim=2, list_elem=None), 's': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TMReg'>, target_dim=3, list_elem=None), 'q0': SymbolBinding(lattice_type=<class 'oqd_core.anal

In [5]:
from oqd_core.analysis.analog import (
    AnalogCFGBuilder,
    AnalogSymbolTableBuilder,
    AnalogTypeChecker,
)
from oqd_core.compiler.analog.passes.compile import compile_analog_circuit

with open("test.analog", mode="r", encoding="utf8") as f:
    source = f.read()

circuit = parse_analog(source)
cfg = AnalogCFGBuilder()(circuit)
cfg = CFGBlockAccumulator()(cfg)
type_checker = AnalogTypeChecker(cfg)
dataflow_result = type_checker.dataflow_result
symbol_analysis = AnalogSymbolTableBuilder(cfg, dataflow_result)
symbol_table = symbol_analysis.symbol_table
circuit, cfg = compile_analog_circuit(circuit, cfg, symbol_table)

In [6]:
from rich import print as pprint

pprint(cfg)

CFG(
    blocks={
        0: CFGBlock(register_id=0, stmts=[], preds=[], succs=[1], exit_nodes=[], edge_labels={}),
        1: CFGBlock(
            register_id=1,
            stmts=[
                Declaration(
                    class_='Declaration',
                    name='r',
                    value=QuantumRegister(class_='QuantumRegister', size=2)
                ),
                Declaration(class_='Declaration', name='s', value=ModeRegister(class_='ModeRegister', size=3)),
                Declaration(
                    class_='Declaration',
                    name='q0',
                    value=Extract(class_='Extract', access=Access(class_='Access', name='r'), index=0)
                ),
                Declaration(
                    class_='Declaration',
                    name='q1',
                    value=Extract(class_='Extract', access=Access(class_='Access', name='r'), index=1)
                ),
                Declaration(
                    class_='Declaration',
                    name='targets',
                    value=AnalogList(
                        class_='AnalogList',
                        values=[Access(class_='Access', name='q0'), Access(class_='Access', name='q1')]
                    )
                ),
                Declaration(class_='Declaration', name='pi', value=MathNum(class_='MathNum', value=3.14159)),
                Declaration(
                    class_='Declaration',
                    name='tau',
                    value=MathMul(
                        class_='MathMul',
                        expr1=MathNum(class_='MathNum', value=2),
                        expr2=Access(class_='Access', name='pi')
                    )
                ),
                Declaration(class_='Declaration', name='omega', value=MathVar(class_='MathVar', name='#omega')),
                Declaration(
                    class_='Declaration',
                    name='phase',
                    value=MathAdd(
                        class_='MathAdd',
                        expr1=MathVar(class_='MathVar', name='#phi'),
                        expr2=MathMul(
                            class_='MathMul',
                            expr1=MathVar(class_='MathVar', name='#t'),
                            expr2=Access(class_='Access', name='omega')
                        )
                    )
                ),
                Declaration(class_='Declaration', name='neg', value=MathNum(class_='MathNum', value=-1)),
                Declaration(class_='Declaration', name='cubed', value=MathNum(class_='MathNum', value=8)),
                Declaration(
                    class_='Declaration',
                    name='sine',
                    value=MathFunc(class_='MathFunc', func='sin', expr=MathNum(class_='MathNum', value=0.5))
                ),
                Declaration(
                    class_='Declaration',
                    name='cosed',
                    value=MathFunc(class_='MathFunc', func='cos', expr=Access(class_='Access', name='phase'))
                ),
                Declaration(
                    class_='Declaration',
                    name='cplx',
                    value=MathAdd(
                        class_='MathAdd',
                        expr1=MathNum(class_='MathNum', value=1.0),
                        expr2=MathMul(
                            class_='MathMul',
                            expr1=MathImag(class_='MathImag'),
                            expr2=MathNum(class_='MathNum', value=0.5)
                        )
                    )
                ),
                Declaration(
                    class_='Declaration',
                    name='X',
                    value=OperatorMul(
                        class_='OperatorMul',
                        op1=MathNum(class_='MathNum', value=1),
                        op2=PauliX(class_='PauliX')
                    )
                ),
                Declaration(class_='Declarati

In [7]:
circuit

AnalogCircuit(class_='AnalogCircuit', statements=[Declaration(class_='Declaration', name='r', value=QuantumRegister(class_='QuantumRegister', size=2)), Declaration(class_='Declaration', name='s', value=ModeRegister(class_='ModeRegister', size=3)), Declaration(class_='Declaration', name='q0', value=Extract(class_='Extract', access=Access(class_='Access', name='r'), index=0)), Declaration(class_='Declaration', name='q1', value=Extract(class_='Extract', access=Access(class_='Access', name='r'), index=1)), Declaration(class_='Declaration', name='targets', value=AnalogList(class_='AnalogList', values=[Access(class_='Access', name='q0'), Access(class_='Access', name='q1')])), Declaration(class_='Declaration', name='pi', value=MathNum(class_='MathNum', value=3.14159)), Declaration(class_='Declaration', name='tau', value=MathMul(class_='MathMul', expr1=MathNum(class_='MathNum', value=2), expr2=Access(class_='Access', name='pi'))), Declaration(class_='Declaration', name='omega', value=MathVar(c

In [8]:
from oqd_compiler_infrastructure import cfg_to_dot

from oqd_core.frontend.analog import serialize_analog

dot = cfg_to_dot(cfg, serialize=serialize_analog)
print(dot.source)
dot.render("cfg", format="png", cleanup=True)


digraph {
	0 [label=" Block #0
------------------------
"]
	0 -> 1
	1 [label=" Block #1
------------------------
r = qreg(2)
s = qmode(3)
q0 = r[0]
q1 = r[1]
targets = [q0, q1]
..."]
	1 -> 26
	26 [label="Branch Block #26
------------------------
Condition: x > 0"]
	26 -> 27
	26 -> 28
	27 [label=" Block #27
------------------------
initialize(r)"]
	27 -> 28
	28 [label="Branch Block #28
------------------------
Condition: x == 1"]
	28 -> 29
	28 -> 30
	29 [label=" Block #29
------------------------
evolve(H_single, 1.0, r[0])"]
	29 -> 31
	30 [label=" Block #30
------------------------
evolve(H_pair, 0.5, r)"]
	30 -> 31
	31 [label=" Block #31
------------------------
n = 3"]
	31 -> 32
	32 [label="Branch Block #32
------------------------
Condition: n > 0"]
	32 -> 33
	32 -> 35
	33 [label=" Block #33
------------------------
evolve(H_rabi, 0.1, targets)
n = -1 + n"]
	33 -> 32
	35 [label="Branch Block #35
------------------------
Condition: true"]
	35 -> 36
	35 -> 38
	36 [label="Branch Block 

'cfg.png'

In [9]:
# from oqd_core.compiler.analog.passes.compile import compile_analog_circuit
# out = compile_analog_circuit(circuit)
# print(printer(out))

In [10]:
# cfg = AnalogCFGBuilder().run(circuit)
# out = json.dumps({node_id: node.to_dict() for node_id, node in cfg.items()}, indent=2)
# print(out)


# print(symbol_table)
# symbol_table = str(symbol_table)
# tree = symbol_table.model_dump_json(indent=2, serialize_as_any=True)
# print(json.dumps(cfg, indent=2))